# Phase 1 — Majority, XLM-R and Gemma baselines



## 1. Environment

In [ ]:
%pip install -q datasets accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 62.0 MB/s eta 0:00:00


In [ ]:
import json
import os
import random
import shutil
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from google.colab import drive
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

drive.mount("/content/drive")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SAVE_DIR = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
PHASE1_DIR = f"{SAVE_DIR}/Phase1"
PHASE2_DIR = f"{SAVE_DIR}/Phase2"
SUMMARY_DIR = f"{PHASE1_DIR}/summary"
XLMR_DIR = f"{PHASE1_DIR}/xlmr_results"
PRED_DIR = f"{PHASE1_DIR}/predictions"
CACHE_DIR = f"{SAVE_DIR}/data_cache"

for directory in [PHASE1_DIR, SUMMARY_DIR, XLMR_DIR, PRED_DIR, CACHE_DIR]:
    os.makedirs(directory, exist_ok=True)

if not os.path.isdir(PHASE2_DIR):
    raise FileNotFoundError(
        f"Complete Phase 2 first. Expected dependency folder: {PHASE2_DIR}"
    )

RUN_MODEL_TRAINING = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_HF_NAME = "brighter-dataset/BRIGHTER-emotion-categories"
MODEL_NAME = "xlm-roberta-large"
LR = 2e-5
NUM_EPOCHS = 5
BATCH_SIZE = 8
MAX_LENGTH = 256

LANGUAGES = {
    "eng": {"name": "English",         "tier": 1, "family": "Indo-European", "subfamily": "Germanic"},
    "hin": {"name": "Hindi",           "tier": 1, "family": "Indo-European", "subfamily": "Indo-Aryan"},
    "rus": {"name": "Russian",         "tier": 1, "family": "Indo-European", "subfamily": "Slavic"},
    "hau": {"name": "Hausa",           "tier": 2, "family": "Afroasiatic",   "subfamily": "Chadic"},
    "kin": {"name": "Kinyarwanda",     "tier": 2, "family": "Niger-Congo",   "subfamily": "Bantu (Great Lakes)"},
    "sun": {"name": "Sundanese",       "tier": 2, "family": "Austronesian",  "subfamily": "Sundic"},
    "yor": {"name": "Yoruba",          "tier": 3, "family": "Niger-Congo",   "subfamily": "Yoruboid"},
    "vmw": {"name": "Emakhuwa",        "tier": 3, "family": "Niger-Congo",   "subfamily": "Bantu (Makua-Lomwe)"},
    "pcm": {"name": "Nigerian Pidgin", "tier": 3, "family": "Creole",        "subfamily": "English-Based"},
}
LANG_ORDER = ["eng", "hin", "rus", "hau", "kin", "sun", "yor", "vmw", "pcm"]
LANG_CODES = LANG_ORDER.copy()
EMOTION_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
ABSENT_EMOTIONS = {"eng": {"disgust"}}

PUBLISHED_BEST = {
    "track_a": {
        "eng": 0.823, "hin": 0.926, "rus": 0.901,
        "hau": 0.751, "kin": 0.657, "sun": 0.550,
        "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
    },
    "track_c": {
        "eng": 0.797, "hin": 0.919, "rus": 0.906,
        "hau": 0.709, "kin": 0.519, "sun": 0.467,
        "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
    },
}
BENCHMARK_SRC_A = "SemEval-2025 Task 11, Table 5 (Track A)"
BENCHMARK_SRC_C = "SemEval-2025 Task 11, Table 7 (Track C)"

if list(LANGUAGES) != LANG_ORDER:
    raise ValueError("LANGUAGES must follow the canonical LANG_ORDER.")


def atomic_json_write(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temporary_path = f"{path}.tmp"
    with open(temporary_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary_path, path)


def save_predictions(phase, lang, condition, y_true, y_pred, emotions):
    path = f"{PRED_DIR}/{phase}_{lang}_{condition}.json"
    atomic_json_write(
        path,
        {
            "emotions": list(emotions),
            "y_true": np.asarray(y_true).astype(int).tolist(),
            "y_pred": np.asarray(y_pred).astype(int).tolist(),
        },
    )


print(f"Device            : {DEVICE}")
print(f"Training enabled  : {RUN_MODEL_TRAINING}")
print(f"Phase 1 directory : {PHASE1_DIR}")
print(f"Languages         : {LANG_ORDER}")

Mounted at /content/drive
Device            : cuda
Training enabled  : False
Phase 1 directory : /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1
Languages         : ['eng', 'hin', 'rus', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']


## 2. Data loading and evaluation

In [ ]:
def get_emotion_cols(df: pd.DataFrame) -> list:
    return [e for e in EMOTION_ORDER if e in df.columns and df[e].fillna(0).sum() > 0]

def load_split(lang_code: str, split: str):
    cpath = f"{CACHE_DIR}/{lang_code}_{split}.parquet"
    if os.path.exists(cpath):
        return pd.read_parquet(cpath)
    try:
        ds = load_dataset(DATASET_HF_NAME, lang_code)
        split_key = split if split in ds else {"validation": "dev", "dev": "validation"}.get(split)
        if not split_key or split_key not in ds:
            return None
        df = ds[split_key].to_pandas()
        for e in EMOTION_ORDER:
            if e not in df.columns:
                df[e] = 0
        df.to_parquet(cpath)
        return df
    except Exception as exc:
        print(f"  Could not load {lang_code}/{split}: {exc}")
        return None

def labels_to_matrix(df: pd.DataFrame, emotion_cols: list) -> np.ndarray:
    result = np.zeros((len(df), len(emotion_cols)), dtype=int)
    for i, e in enumerate(emotion_cols):
        if e in df.columns:
            result[:, i] = df[e].fillna(0).astype(int).values
    return result



def get_eval_emotions(lang_code: str | None,
                      emotion_cols: list[str]) -> list[str]:
    """
    Return the emotion labels that must contribute to macro-F1.

    English excludes disgust because it is structurally absent.
    Every other language retains all supplied labels.
    """
    if lang_code is None:
        return list(emotion_cols)

    absent = ABSENT_EMOTIONS.get(lang_code, set())
    return [emotion for emotion in emotion_cols if emotion not in absent]


def macro_f1(y_true: np.ndarray,
             y_pred: np.ndarray,
             emotion_cols: list[str],
             lang_code: str | None = None) -> dict:
    """
    Calculate per-emotion F1 for every stored column.

    Macro-F1 uses five labels for English and all supplied labels
    for every other language.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    if y_true.shape[1] != len(emotion_cols):
        raise ValueError(
            f"Expected {len(emotion_cols)} columns but received "
            f"{y_true.shape[1]}"
        )

    per_label = {}
    unrounded_scores = {}

    # Keep every per-emotion result, including English disgust = 0.
    for index, label in enumerate(emotion_cols):
        score = f1_score(
            y_true[:, index],
            y_pred[:, index],
            zero_division=0,
        )
        unrounded_scores[label] = float(score)
        per_label[label] = round(float(score), 4)

    eval_emotions = get_eval_emotions(lang_code, emotion_cols)

    if not eval_emotions:
        raise ValueError(f"No evaluation emotions available for {lang_code}")

    per_label["macro_f1"] = round(
        float(np.mean([unrounded_scores[e] for e in eval_emotions])),
        4,
    )

    return per_label

print("Loading BRIGHTER data ...")
DATA = {}
for code in LANG_ORDER:
    DATA[code] = {}
    for split in ["train", "validation", "test"]:
        dataframe = load_split(code, split)
        if dataframe is not None:
            DATA[code][split] = dataframe

    required_splits = {"train", "validation", "test"}
    missing_splits = sorted(required_splits - set(DATA[code]))
    if missing_splits:
        raise RuntimeError(f"{code}: missing dataset splits {missing_splits}")

    active = get_emotion_cols(DATA[code]["train"])
    print(
        f"  {LANGUAGES[code]['name']:18s} "
        f"train={len(DATA[code]['train']):5d} "
        f"validation={len(DATA[code]['validation']):5d} "
        f"test={len(DATA[code]['test']):5d} "
        f"emotions={active}"
    )

print("Dataset loading complete.")

Loading BRIGHTER data ...
  English            train= 2764 validation=  230 test= 5528 emotions=['anger', 'fear', 'joy', 'sadness', 'surprise']
  Hindi              train= 2556 validation=  200 test= 2020 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Russian            train= 2679 validation=  398 test= 2000 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Hausa              train= 2145 validation=  712 test= 2160 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Kinyarwanda        train= 2451 validation=  814 test= 2462 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Sundanese          train=  924 validation=  398 test= 1852 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Yoruba             train= 2992 validation=  994 test= 3000 emotions=['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
  Emakhuwa           train= 1551 validation=  516 test= 1554 emotions=['anger', 'd

## 3. Baseline 1 — optimised majority classifier

In [ ]:
def find_optimal_majority(source_df: pd.DataFrame,
                          eval_df: pd.DataFrame,
                          emotion_cols: list,
                          lang_code: str | None = None) -> dict:
    """
    Try predicting top-1, top-2 ... top-N most frequent emotions.
    Return prediction dict that maximises macro F1 on eval_df.
    """
    freqs       = {e: float(source_df[e].fillna(0).mean()) for e in emotion_cols}
    sorted_emos = sorted(freqs.items(), key=lambda x: x[1], reverse=True)
    y_eval      = labels_to_matrix(eval_df, emotion_cols)
    best_f1, best_n = -1.0, 1

    for top_n in range(1, len(emotion_cols) + 1):
        top_set  = {e for e, _ in sorted_emos[:top_n]}
        pred_vec = np.array([1 if e in top_set else 0 for e in emotion_cols])
        y_pred   = np.tile(pred_vec, (len(y_eval), 1))
        f        = macro_f1(y_eval, y_pred, emotion_cols, lang_code=lang_code,)["macro_f1"]
        if f > best_f1:
            best_f1, best_n = f, top_n

    top_set = {e for e, _ in sorted_emos[:best_n]}
    return {e: (1 if e in top_set else 0) for e in emotion_cols}


print("\n" + "="*60)
print("BASELINE 1 — MAJORITY CLASS (Optimised Top-N)")
print("="*60)
majority_results: dict = {"track_a": {}, "track_c": {}}

for code in LANG_CODES:
    # Track A: dynamic emotions from target train split
    ecols_a   = get_emotion_cols(DATA[code]["train"])
    test_y_a  = labels_to_matrix(DATA[code]["test"], ecols_a)
    pred_dict = find_optimal_majority(
        DATA[code]["train"], DATA[code]["validation"], ecols_a, lang_code=code)
    pred_vec  = np.array([pred_dict[e] for e in ecols_a])
    y_pred_a  = np.tile(pred_vec, (len(test_y_a), 1))
    majority_results["track_a"][code] = macro_f1(test_y_a, y_pred_a, ecols_a, lang_code=code)

    # Track C: EMOTION_ORDER, use test distribution
    test_y_c    = labels_to_matrix(DATA[code]["test"], EMOTION_ORDER)
    pred_dict_c = find_optimal_majority(
        DATA[code]["test"], DATA[code]["test"], EMOTION_ORDER, lang_code=code)
    pred_vec_c  = np.array([pred_dict_c[e] for e in EMOTION_ORDER])
    y_pred_c    = np.tile(pred_vec_c, (len(test_y_c), 1))
    majority_results["track_c"][code] = macro_f1(test_y_c, y_pred_c, EMOTION_ORDER,  lang_code=code)

    name  = LANGUAGES[code]["name"]
    f1_a  = majority_results["track_a"][code]["macro_f1"]
    f1_c  = majority_results["track_c"][code]["macro_f1"]
    print(f"  {name:20s}  Track A: {f1_a:.4f} ({len(ecols_a)} emo)"
          f"  |  Track C: {f1_c:.4f} "
          f"({len(get_eval_emotions(code, EMOTION_ORDER))} emo)")

print("\n Majority class complete.")


BASELINE 1 — MAJORITY CLASS (Optimised Top-N)
  English               Track A: 0.4491 (5 emo)  |  Track C: 0.4491 (5 emo)
  Hindi                 Track A: 0.2641 (6 emo)  |  Track C: 0.2641 (6 emo)
  Russian               Track A: 0.2618 (6 emo)  |  Track C: 0.2618 (6 emo)
  Hausa                 Track A: 0.3121 (6 emo)  |  Track C: 0.3121 (6 emo)
  Kinyarwanda           Track A: 0.2176 (6 emo)  |  Track C: 0.2176 (6 emo)
  Sundanese             Track A: 0.3340 (6 emo)  |  Track C: 0.3340 (6 emo)
  Yoruba                Track A: 0.1647 (6 emo)  |  Track C: 0.1647 (6 emo)
  Emakhuwa              Track A: 0.1626 (6 emo)  |  Track C: 0.1626 (6 emo)
  Nigerian Pidgin       Track A: 0.3566 (6 emo)  |  Track C: 0.3566 (6 emo)

✓ Majority class complete.


## 4. Baseline 2 — XLM-RoBERTa-large

In [ ]:
os.environ["HF_HOME"] = "/content/hf_cache_local"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache_local/hub"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
os.makedirs("/content/hf_cache_local/hub", exist_ok=True)

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, emotion_cols, max_len=MAX_LENGTH):
        self.texts   = df["text"].tolist()
        self.labels  = df[emotion_cols].fillna(0).astype(float).values
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float),
        }


# ── 5b: Model ────────────────────────────────────────────────────────────────
class XLMRoBERTaMultiLabel(nn.Module):
    def __init__(self, model_name, n_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, cache_dir="/content/hf_cache_local")
        hidden          = self.encoder.config.hidden_size
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.drop(out.last_hidden_state[:, 0, :])
        return self.classifier(cls)


# ── 5d: Train function ───────────────────────────────────────────────────────
def train_model(train_df, dev_df, tokenizer, emotion_cols,
                model_name="xlm-roberta-large",
                epochs=5, batch_size=BATCH_SIZE, lr=2e-5,
                max_len=MAX_LENGTH, patience=2, pos_weight=None):

    train_dl = DataLoader(
        EmotionDataset(train_df, tokenizer, emotion_cols, max_len),
        batch_size=batch_size, shuffle=True)
    dev_dl   = DataLoader(
        EmotionDataset(dev_df, tokenizer, emotion_cols, max_len),
        batch_size=batch_size * 2)

    model   = XLMRoBERTaMultiLabel(model_name, n_labels=len(emotion_cols)).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE) if pos_weight is not None else None
    )

    opt   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(
        opt,
        num_warmup_steps=int(0.1 * len(train_dl) * epochs),
        num_training_steps=len(train_dl) * epochs,
    )

    best_f1    = 0.0
    no_improve = 0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    for ep in range(1, epochs + 1):
        model.train(); total_loss = 0.0
        for batch in train_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labs = batch["labels"].to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(ids, mask), labs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            total_loss += loss.item()

        model.eval(); preds_list, true_list = [], []
        with torch.no_grad():
            for batch in dev_dl:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                p    = (torch.sigmoid(model(ids, mask)) > 0.5).int().cpu().numpy()
                preds_list.append(p)
                true_list.append(batch["labels"].int().numpy())

        dev_f1   = macro_f1(np.vstack(true_list), np.vstack(preds_list), emotion_cols)["macro_f1"]
        avg_loss = total_loss / len(train_dl)
        print(f"    Epoch {ep}/{epochs}  loss={avg_loss:.4f}  dev_F1={dev_f1:.4f}")

        if dev_f1 > best_f1:
            best_f1    = dev_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"    Early stop (patience={patience}).")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    return model

def evaluate_model(model, test_df, tokenizer, emotion_cols,
                   batch_size=32, max_len=MAX_LENGTH,
                   run_tag=None, lang_code=None):
    test_dl = DataLoader(
        EmotionDataset(test_df, tokenizer, emotion_cols, max_len),
        batch_size=batch_size)
    model.eval(); preds_list, true_list = [], []
    with torch.no_grad():
        for batch in test_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            p    = (torch.sigmoid(model(ids, mask)) > 0.5).int().cpu().numpy()
            preds_list.append(p)
            true_list.append(batch["labels"].int().numpy())
    y_true_all, y_pred_all = np.vstack(true_list), np.vstack(preds_list)
    if run_tag is not None:
        save_predictions(run_tag[0], run_tag[1], run_tag[2],
                         y_true_all, y_pred_all, emotion_cols)
    return macro_f1(y_true_all, y_pred_all, emotion_cols, lang_code=lang_code)

In [ ]:
TRACK_A_PATH = f"{XLMR_DIR}/xlmr_track_a.json"
TRACK_C_PATH = f"{XLMR_DIR}/xlmr_track_c.json"
POS_WEIGHT_LANGS = {"kin", "yor", "vmw"}


def load_result_map(path):
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise TypeError(f"{path}: expected a JSON object")
    return payload


xlmr_results = {
    "track_a": load_result_map(TRACK_A_PATH),
    "track_c": load_result_map(TRACK_C_PATH),
}

if RUN_MODEL_TRAINING:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        cache_dir="/content/hf_cache_local",
    )
else:
    tokenizer = None

print("Loaded completed XLM-R conditions:")
for track in ["track_a", "track_c"]:
    print(f"  {track}: {sorted(xlmr_results[track])}")

Loaded completed XLM-R conditions:
  track_a: ['eng', 'hau', 'hin', 'kin', 'pcm', 'rus', 'sun', 'vmw', 'yor']
  track_c: ['eng', 'hau', 'hin', 'kin', 'pcm', 'rus', 'sun', 'vmw', 'yor']


### 4.1 Track A — monolingual fine-tuning

In [ ]:
if RUN_MODEL_TRAINING:
    print("Track A: monolingual fine-tuning")

    for code in LANG_ORDER:
        if code in xlmr_results["track_a"]:
            print(f"  Skipping {LANGUAGES[code]['name']}: result already exists")
            continue

        lang_seed = RANDOM_SEED + LANG_ORDER.index(code)
        random.seed(lang_seed)
        np.random.seed(lang_seed)
        torch.manual_seed(lang_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(lang_seed)

        emotion_cols = get_emotion_cols(DATA[code]["train"])
        pos_weight = None

        if code in POS_WEIGHT_LANGS:
            train_df = DATA[code]["train"]
            positive = train_df[emotion_cols].fillna(0).sum()
            negative = len(train_df) - positive
            pos_weight = torch.tensor(
                (negative / positive.clip(lower=1)).clip(upper=10).values,
                dtype=torch.float,
            )

        print(
            f"  {code.upper()} {LANGUAGES[code]['name']} "
            f"seed={lang_seed} "
            f"loss={'weighted BCE' if pos_weight is not None else 'BCE'}"
        )

        model = train_model(
            DATA[code]["train"],
            DATA[code]["validation"],
            tokenizer,
            emotion_cols,
            model_name=MODEL_NAME,
            epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_len=MAX_LENGTH,
            patience=2,
            pos_weight=pos_weight,
        )
        result = evaluate_model(
            model,
            DATA[code]["test"],
            tokenizer,
            emotion_cols,
            run_tag=("phase1_xlmr_a", code, "mono"),
            lang_code=code,
        )
        xlmr_results["track_a"][code] = result
        atomic_json_write(TRACK_A_PATH, xlmr_results["track_a"])

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"    macro-F1={result['macro_f1']:.4f}")
else:
    print("Track A training skipped; completed results will be validated below.")

Track A training skipped; completed results will be validated below.


### 4.2 Track C — leave-one-out cross-lingual fine-tuning

In [ ]:
if RUN_MODEL_TRAINING:
    print("Track C: leave-one-out cross-lingual fine-tuning")

    for code in LANG_ORDER:
        if code in xlmr_results["track_c"]:
            print(f"  Skipping {LANGUAGES[code]['name']}: result already exists")
            continue

        source_codes = [source for source in LANG_ORDER if source != code]
        train_df = pd.concat(
            [DATA[source]["train"] for source in source_codes],
            ignore_index=True,
        )
        validation_df = pd.concat(
            [DATA[source]["validation"] for source in source_codes],
            ignore_index=True,
        )
        test_df = DATA[code]["test"]

        lang_seed = RANDOM_SEED + LANG_ORDER.index(code)
        random.seed(lang_seed)
        np.random.seed(lang_seed)
        torch.manual_seed(lang_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(lang_seed)

        print(
            f"  {code.upper()} {LANGUAGES[code]['name']} "
            f"sources={source_codes} seed={lang_seed}"
        )

        model = train_model(
            train_df,
            validation_df,
            tokenizer,
            EMOTION_ORDER,
            model_name=MODEL_NAME,
            epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_len=MAX_LENGTH,
            patience=4,
            pos_weight=None,
        )
        result = evaluate_model(
            model,
            test_df,
            tokenizer,
            EMOTION_ORDER,
            run_tag=("phase1_xlmr_c", code, "loo"),
            lang_code=code,
        )
        xlmr_results["track_c"][code] = result
        atomic_json_write(TRACK_C_PATH, xlmr_results["track_c"])

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"    macro-F1={result['macro_f1']:.4f}")
else:
    print("Track C training skipped; completed results will be validated below.")

for track, path in [("track_a", TRACK_A_PATH), ("track_c", TRACK_C_PATH)]:
    missing = [code for code in LANG_ORDER if code not in xlmr_results[track]]
    if missing:
        raise ValueError(f"{track}: missing XLM-R results for {missing}")
    atomic_json_write(path, xlmr_results[track])

print("XLM-R Track A and Track C results are complete.")

Track C training skipped; completed results will be validated below.
XLM-R Track A and Track C results are complete.


## 5. Baseline 3 — canonical Gemma-4 P0

In [ ]:
PHASE2_RESULTS_PATH = f"{PHASE2_DIR}/phase2_validation_raw_results_gemma4.json"
CANONICAL_P0_AUDIT_PATH = (
    f"{SUMMARY_DIR}/phase1_gemma_p0_canonical_from_phase2.json"
)

if not os.path.exists(PHASE2_RESULTS_PATH):
    raise FileNotFoundError(PHASE2_RESULTS_PATH)

with open(PHASE2_RESULTS_PATH, encoding="utf-8") as handle:
    phase2_results = json.load(handle)

required_score_keys = set(EMOTION_ORDER + ["macro_f1"])
canonical_p0 = {}

for code in LANG_ORDER:
    result = phase2_results.get(code, {}).get("p0")
    if not isinstance(result, dict):
        raise ValueError(f"{code}: canonical Phase 2 P0 result is missing")
    missing_keys = sorted(required_score_keys - set(result))
    if missing_keys:
        raise ValueError(f"{code}: P0 is missing score keys {missing_keys}")
    canonical_p0[code] = deepcopy(result)

gemma4_results = {
    "track_a": deepcopy(canonical_p0),
    "track_c": deepcopy(canonical_p0),
}

atomic_json_write(
    CANONICAL_P0_AUDIT_PATH,
    {
        "source": PHASE2_RESULTS_PATH,
        "condition": "p0",
        "role": "Phase 1 B3 baseline and Phase 4 neither condition",
        "results": canonical_p0,
    },
)

print("Canonical Phase 2 P0 imported as the Phase 1 B3 baseline.")
for code in LANG_ORDER:
    print(
        f"  {code.upper()} {LANGUAGES[code]['name']:18s} "
        f"P0={canonical_p0[code]['macro_f1']:.4f}"
    )

Canonical Phase 2 P0 imported as the Phase 1 B3 baseline.
  ENG English            P0=0.6274
  HIN Hindi              P0=0.7421
  RUS Russian            P0=0.8180
  HAU Hausa              P0=0.5606
  KIN Kinyarwanda        P0=0.4213
  SUN Sundanese          P0=0.6516
  YOR Yoruba             P0=0.3479
  VMW Emakhuwa           P0=0.0209
  PCM Nigerian Pidgin    P0=0.5199


## 6. Compile, validate and save Phase 1 outputs

In [ ]:
def build_summary_table(track):
    _is_a     = (track == "track_a")
    _tl       = "Track A" if _is_a else "Track C"
    _pub_col  = "Published Track A best macro-F1" if _is_a else "Published Track C best macro-F1"
    _gap_xlmr = f"Gap: XLM-R minus published {_tl} best"
    _gap_g4   = f"Gap: Gemma-4-31B minus published {_tl} best"
    _src      = BENCHMARK_SRC_A if _is_a else BENCHMARK_SRC_C
    rows = []
    for code in LANG_ORDER:
        base_emotions = (
            get_emotion_cols(DATA[code]["train"])
            if track == "track_a"
            else EMOTION_ORDER
        )
        evaluation_emotions = get_eval_emotions(code, base_emotions)
        info     = LANGUAGES[code]
        xlmr_f1  = xlmr_results[track][code]["macro_f1"]
        g4_f1    = gemma4_results[track][code]["macro_f1"]
        pub_best = PUBLISHED_BEST[track][code]
        rows.append({
            "Language":                       info["name"],
            "Language code":                  code.upper(),
            "Resource tier":                  info["tier"],
            "Linguistic subgroup":            info["subfamily"],
            "Number of evaluated emotions":   len(evaluation_emotions),
            "Majority baseline macro-F1":     majority_results[track][code]["macro_f1"],
            "XLM-R macro-F1":                 xlmr_f1,
            "Gemma-4-31B zero-shot macro-F1": g4_f1,
            _pub_col:                         pub_best,
            "Evaluation track":               _tl,
            _gap_xlmr:                        round(xlmr_f1 - pub_best, 4),
            _gap_g4:                          round(g4_f1 - pub_best, 4),
            "Published benchmark source":     _src,
        })
    return pd.DataFrame(rows)


def build_detail_table():
    rows = []
    for track in ["track_a", "track_c"]:
        _is_a  = (track == "track_a")
        _tl    = "Track A" if _is_a else "Track C"
        _src   = BENCHMARK_SRC_A if _is_a else BENCHMARK_SRC_C
        for code in LANG_ORDER:
            base_emotions = (
                get_emotion_cols(DATA[code]["train"])
                if track == "track_a"
                else EMOTION_ORDER
            )
            evaluation_emotions = get_eval_emotions(code, base_emotions)
            pub_best = PUBLISHED_BEST[track][code]
            for system_name, result in [
                ("B1-Majority",   majority_results[track][code]),
                ("B2-XLM-R",      xlmr_results[track][code]),
                ("B3-Gemma4-31B", gemma4_results[track][code]),
            ]:
                macro_f1 = result.get("macro_f1")
                rows.append({
                    "Evaluation track":             _tl,
                    "Language code":                code.upper(),
                    "Language":                     LANGUAGES[code]["name"],
                    "Resource tier":                LANGUAGES[code]["tier"],
                    "System":                       system_name,
                    "Number of evaluated emotions": len(evaluation_emotions),
                    "Anger F1":                     result.get("anger"),
                    "Fear F1":                      result.get("fear"),
                    "Joy F1":                       result.get("joy"),
                    "Sadness F1":                   result.get("sadness"),
                    "Surprise F1":                  result.get("surprise"),
                    "Macro-F1":                     macro_f1,
                    "Disgust F1":                   result.get("disgust"),
                    "Published best macro-F1 for the same track": pub_best,
                    "Gap to same-track published best (system minus published)": round(macro_f1 - pub_best, 4) if macro_f1 is not None else None,
                    "Published benchmark source":   _src,
                })
    return pd.DataFrame(rows)


track_a_summary = build_summary_table("track_a")
track_c_summary = build_summary_table("track_c")
detailed_results = build_detail_table()

track_a_summary.to_csv(
    f"{SUMMARY_DIR}/phase1_track_a_summary_gemma4.csv",
    index=False,
)
track_c_summary.to_csv(
    f"{SUMMARY_DIR}/phase1_track_c_summary_gemma4.csv",
    index=False,
)
detailed_results.to_csv(
    f"{SUMMARY_DIR}/phase1_detailed_gemma4.csv",
    index=False,
)

summary_path = f"{SUMMARY_DIR}/phase1_raw_results_gemma4.json"
existing_summary = {}
if os.path.exists(summary_path):
    with open(summary_path, encoding="utf-8") as handle:
        existing_summary = json.load(handle)

summary_payload = {
    "majority": majority_results,
    "xlmr": xlmr_results,
    "gemma4_31b": gemma4_results,
    "canonical_p0_source": PHASE2_RESULTS_PATH,
}
if "serengeti" in existing_summary:
    summary_payload["serengeti"] = existing_summary["serengeti"]

atomic_json_write(summary_path, summary_payload)

assert len(track_a_summary) == len(LANG_ORDER)
assert len(track_c_summary) == len(LANG_ORDER)
assert len(detailed_results) == 54
assert track_a_summary.loc[
    track_a_summary["Language code"] == "ENG", "Number of evaluated emotions"
].item() == 5
assert track_c_summary.loc[
    track_c_summary["Language code"] == "ENG", "Number of evaluated emotions"
].item() == 5

print("Track A")
display(track_a_summary)
print("Track C")
display(track_c_summary)
print(f"Saved Phase 1 outputs to: {SUMMARY_DIR}")

Track A


,Language,Language code,Resource tier,Linguistic subgroup,Number of evaluated emotions,Majority baseline macro-F1,XLM-R macro-F1,Gemma-4-31B zero-shot macro-F1,Published Track A best macro-F1,Evaluation track,Gap: XLM-R minus published Track A best,Gap: Gemma-4-31B minus published Track A best,Published benchmark source
0,English,ENG,1,Germanic,5,0.4491,0.5523,0.6274,0.823,Track A,-0.2707,-0.1956,"SemEval-2025 Task 11, Table 5 (Track A)"
1,Hindi,HIN,1,Indo-Aryan,6,0.2641,0.8550,0.7421,0.926,Track A,-0.0710,-0.1839,"SemEval-2025 Task 11, Table 5 (Track A)"
2,Russian,RUS,1,Slavic,6,0.2618,0.8791,0.8180,0.901,Track A,-0.0219,-0.0830,"SemEval-2025 Task 11, Table 5 (Track A)"
3,Hausa,HAU,2,Chadic,6,0.3121,0.6326,0.5606,0.751,Track A,-0.1184,-0.1904,"SemEval-2025 Task 11, Table 5 (Track A)"
4,Kinyarwanda,KIN,2,Bantu (Great Lakes),6,0.2176,0.3928,0.4213,0.657,Track A,-0.2642,-0.2357,"SemEval-2025 Task 11, Table 5 (Track A)"
5,Sundanese,SUN,2,Sundic,6,0.3340,0.2026,0.6516,0.550,Track A,-0.3474,0.1016,"SemEval-2025 Task 11, Table 5 (Track A)"
6,Yoruba,YOR,3,Yoruboid,6,0.1647,0.1267,0.3479,0.461,Track A,-0.3343,-0.1131,"SemEval-2025 Task 11, Table 5 (Track A)"
7,Emakhuwa,VMW,3,Bantu (Makua-Lomwe),6,0.1626,0.0380,0.0209,0.325,Track A,-0.2870,-0.3041,"SemEval-2025 Task 11, Table 5 (Track A)"
8,Nigerian Pidgin,PCM,3,English-Based,6,0.3566,0.5692,0.5199,0.674,Track A,-0.1048,-0.1541,"SemEval-2025 Task 11, Table 5 (Track A)"


Track C


,Language,Language code,Resource tier,Linguistic subgroup,Number of evaluated emotions,Majority baseline macro-F1,XLM-R macro-F1,Gemma-4-31B zero-shot macro-F1,Published Track C best macro-F1,Evaluation track,Gap: XLM-R minus published Track C best,Gap: Gemma-4-31B minus published Track C best,Published benchmark source
0,English,ENG,1,Germanic,5,0.4491,0.4217,0.6274,0.797,Track C,-0.3753,-0.1696,"SemEval-2025 Task 11, Table 7 (Track C)"
1,Hindi,HIN,1,Indo-Aryan,6,0.2641,0.7359,0.7421,0.919,Track C,-0.1831,-0.1769,"SemEval-2025 Task 11, Table 7 (Track C)"
2,Russian,RUS,1,Slavic,6,0.2618,0.6601,0.8180,0.906,Track C,-0.2459,-0.0880,"SemEval-2025 Task 11, Table 7 (Track C)"
3,Hausa,HAU,2,Chadic,6,0.3121,0.1426,0.5606,0.709,Track C,-0.5664,-0.1484,"SemEval-2025 Task 11, Table 7 (Track C)"
4,Kinyarwanda,KIN,2,Bantu (Great Lakes),6,0.2176,0.0967,0.4213,0.519,Track C,-0.4223,-0.0977,"SemEval-2025 Task 11, Table 7 (Track C)"
5,Sundanese,SUN,2,Sundic,6,0.3340,0.3355,0.6516,0.467,Track C,-0.1315,0.1846,"SemEval-2025 Task 11, Table 7 (Track C)"
6,Yoruba,YOR,3,Yoruboid,6,0.1647,0.0912,0.3479,0.359,Track C,-0.2678,-0.0111,"SemEval-2025 Task 11, Table 7 (Track C)"
7,Emakhuwa,VMW,3,Bantu (Makua-Lomwe),6,0.1626,0.0440,0.0209,0.210,Track C,-0.1660,-0.1891,"SemEval-2025 Task 11, Table 7 (Track C)"
8,Nigerian Pidgin,PCM,3,English-Based,6,0.3566,0.3487,0.5199,0.674,Track C,-0.3253,-0.1541,"SemEval-2025 Task 11, Table 7 (Track C)"


Saved Phase 1 outputs to: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/summary
